This notebook will demonstrate the implementation of an alternate algorithm in the `cell_type_mapper`. This algorithm was developed for use by some internal science groups at the Allen Institute. Documentation for this algorithm is, at this point, extremely limited. Proceed at your own risk.

# Download data necessary to run cell_type_mapper

In [1]:
import abc_atlas_access.abc_atlas_cache.abc_project_cache as abc_project_cache

In [2]:
import anndata
import numpy as np
import os
import pandas as pd
import pathlib

dst_dir = "/Volumes/mazarbul/workspace/cell_type_mapper_demos"
cache_dir = "/Volumes/mazarbul/workspace/abc_cache"

if not os.path.isdir(cache_dir):
    raise RuntimeError(
        f"{cache_dir} is not a valid directory in your system\n"
        "Set cache_dir above to the path to the directory where you want to download ABC Atlas data"
    )

if not os.path.isdir(dst_dir):
    raise RuntimeError(
        f"{dst_dir} is not a valid directory in your system\n"
        "Set dst_dir above to the path to the directory where you want to download ABC Atlas data"
    )


abc_cache = abc_project_cache.AbcProjectCache.from_cache_dir(cache_dir)

In [3]:
gene_mapper_db_path = abc_cache.get_file_path(
    directory="mmc-gene-mapper",
    file_name="mmc_gene_mapper.2025-08-04"
)

In [4]:
whb_marker_path = abc_cache.get_file_path(
    directory="WHB-taxonomy",
    file_name="query_markers.n10.20240221800"
)


In [5]:
whb_precomputed_path = abc_cache.get_file_path(
    directory="WHB-taxonomy",
    file_name="precomputed_stats.siletti.training"
)



In [6]:
raw_query_path = abc_cache.get_file_path(
    directory="WMB-10Xv3",
    file_name="WMB-10Xv3-P/raw"
)

# only map a subset of cells
src = anndata.read_h5ad(raw_query_path, backed='r')
src.strings_to_categoricals()
query_path = pathlib.Path(dst_dir) / 'alt_algorithm_query.h5ad'
subset = src[:5000]
subset.write_h5ad(query_path)
src.file.close()
del src

# Algorithm description

Where as the default `cell_type_mapper` algorithm conducts independent bootstrapped mappings at each level of the taxonomy tree, the new algorithm conducts independent bootstrapped mappings to the entire taxonomy.

Consider a taxonomy with three levels: "class", "subclass", and "cluster"

### default algorithm (type_selection.algorithm = "hierarchical" in config)

1) Map to class 100 times using a different random selection of marker genes each time. Select the class that was chosen most frequently.
2) Map to subclass 100 times using a different random selection of marker genes each time and only considering the subclasses descended from the class chosen in (1). Select the subclass that was chosen most frequently.
3) Map to cluster 100 times using a different random selection of marker genes each time and only considering the clusters descended from the subclass chosen in (2).

### new algorithm (type_selection.algorithm = "hann" in config)

1) Randomly subset marker genes. Map to class once.
2) Randomly subset marker genes. Map to subclass once, only considering subclasses descended from class chosen in (1).
3) Randomly subset marker genes. Map to cluster once, only considering clusters descended from subclass chosen in (2).
4) Repeat (1)-(3) 100 times. Record how often each cluster was chosen in (3).


# Run cell_type_mapper with alternative algorithm specified

To generate a low quality mapping (so that we can show how uncertainty is recorded in the new algorithm), we will map data from the Yao et al. 2023 Whole Mouse Brain dataset onto the Whole Human Brain taxonomy defined in Siletti et al. 2023.

In [7]:
from cell_type_mapper.cli.from_specified_markers import FromSpecifiedMarkersRunner

In [8]:
dst_path = pathlib.Path(dst_dir) / "alt_algorithm_mapping.h5"

In [9]:
config = {
    "query_path": str(query_path),
    "gene_mapping": {"db_path": str(gene_mapper_db_path)},
    "query_markers": {"serialized_lookup": str(whb_marker_path)},
    "precomputed_stats": {"path": str(whb_precomputed_path)},
    "hdf5_result_path": str(dst_path),
    "drop_level": "CCN20230722_SUPT",
    "type_assignment": {
        "normalization": "raw",
        "n_processors": 4,
        "chunk_size": 500,
        "algorithm": "hann",  # this is where we specify the new algorithm
        "bootstrap_factor": 0.5,
        "bootstrap_iteration": 100
    }
}

In [10]:
%%time
runner = FromSpecifiedMarkersRunner(
    args=[],
    input_data=config
)
runner.run()

=== Running HANN Mapping 1.7.0rc with config ===
{
  "obsm_clobber": false,
  "max_gb": 100.0,
  "query_gene_id_col": null,
  "query_path": "/Volumes/mazarbul/workspace/cell_type_mapper_demos/alt_algorithm_query.h5ad",
  "gene_mapping": {
    "log_level": "ERROR",
    "db_path": "/Volumes/mazarbul/workspace/abc_cache/mapmycells/mmc-gene-mapper/20250630/mmc_gene_mapper.2025-08-04.db"
  },
  "csv_result_path": null,
  "drop_level": "CCN20230722_SUPT",
  "tmp_dir": null,
  "log_path": null,
  "hdf5_result_path": "/Volumes/mazarbul/workspace/cell_type_mapper_demos/alt_algorithm_mapping.h5",
  "nodes_to_drop": null,
  "verbose_stdout": true,
  "extended_result_dir": null,
  "precomputed_stats": {
    "log_level": "ERROR",
    "path": "/Volumes/mazarbul/workspace/abc_cache/mapmycells/WHB-taxonomy/20240831/precomputed_stats.siletti.training.h5"
  },
  "obsm_key": null,
  "extended_result_path": null,
  "log_level": "ERROR",
  "summary_metadata_path": null,
  "map_to_ensembl": false,
  "type_a

/Users/scott.daniel/KnowledgeEngineering/cell_type_mapper/src/cell_type_mapper/taxonomy/utils.py:253: UserWarning: This taxonomy has no mapping from leaf_node -> rows in the cell by gene matrix
  warnings.warn("This taxonomy has no mapping from leaf_node -> rows "


***Checking to see if we need to map query genes onto reference dataset
====Based on 59357 genes, your input data is from species 'Homo sapiens Linnaeus, 1758:9606'
Reference data belongs to species Homo sapiens Linnaeus, 1758:9606
Reference genes are from authority 'ENSEMBL'
Mapping input genes to 'Homo sapiens Linnaeus, 1758:9606 -- ENSEMBL' using
http://github.com/AllenInstitute/mmc_gene_mapper version 0.2.1
backed by database file: mmc_gene_mapper.2025-08-04.db
created on: 2025-08-04-18-10-52
hash: md5:755b0724c2ff00cc199f48e2718a09e5
Based on 32285 genes, your input data is from species 'Balb/c mouse:10090'
Input genes are from species 'Balb/c mouse:10090'
Mapping input genes from 'ENSEMBL' to 'NCBI'
Mapping genes from species 'Balb/c mouse:10090' to 'Homo sapiens Linnaeus, 1758:9606'
Mapping input genes from 'NCBI' to 'ENSEMBL'
***Mapping of query genes to reference dataset complete
BENCHMARK: spent 1.2650e-01 seconds creating query marker cache
Scanning unlabeled data to check t

/Users/scott.daniel/KnowledgeEngineering/cell_type_mapper/src/cell_type_mapper/cli/cli_log.py:73: UserWarning: parent node 'CCN202210140_CLUS/CS202210140_80' had too few markers in query set; augmenting with markers from ['CCN202210140_SUPC/CS202210140_472']
  warnings.warn(msg)
/Users/scott.daniel/KnowledgeEngineering/cell_type_mapper/src/cell_type_mapper/cli/cli_log.py:73: UserWarning: parent node 'CCN202210140_CLUS/CS202210140_76' had too few markers in query set; augmenting with markers from ['CCN202210140_SUPC/CS202210140_468']
  warnings.warn(msg)
/Users/scott.daniel/KnowledgeEngineering/cell_type_mapper/src/cell_type_mapper/cli/cli_log.py:73: UserWarning: parent node 'CCN202210140_CLUS/CS202210140_71' had too few markers in query set; augmenting with markers from ['CCN202210140_SUPC/CS202210140_471']
  warnings.warn(msg)
/Users/scott.daniel/KnowledgeEngineering/cell_type_mapper/src/cell_type_mapper/cli/cli_log.py:73: UserWarning: parent node 'CCN202210140_CLUS/CS202210140_66' ha

Chunking through query data
2000 of 5000 cells in 1.78e+01 sec; predict 2.67e+01 sec of 4.46e+01 sec left
2500 of 5000 cells in 2.57e+01 sec; predict 2.57e+01 sec of 5.15e+01 sec left
3000 of 5000 cells in 2.74e+01 sec; predict 1.82e+01 sec of 4.56e+01 sec left
3500 of 5000 cells in 2.90e+01 sec; predict 1.24e+01 sec of 4.14e+01 sec left
BENCHMARK: spent 4.0637e+01 seconds assigning cell types
Writing marker genes to output file
MAPPING FROM SPECIFIED MARKERS RAN SUCCESSFULLY
WRITING OUTPUT AND CLEANING UP
CPU times: user 36.4 s, sys: 20.5 s, total: 56.9 s
Wall time: 1min 7s


# Examine output

The alternative output produces a single HDF5 file.

The HDF5 file contains two numpy arrays encoding the results of the mapping, `votes` and `correlation`.

`votes` is an array of integers that of shape (n_cells, n_clusters). The values represent the number of times that a given cell mapped to a given cluster (leaf node) in the taxonomy.

`correlation` is an array of floats of shape (n_cells, n_clusters). It represents the correlation of the cell with the cluster averaged over the mapping iterations that chose that cluster.

Rows and columns can be mapped to cells and clusters using the `cell_identifiers` and `cluster_identifiers` arrays.

In [11]:
import h5py
import json

In [12]:
mapping = h5py.File(dst_path, 'r')

In [13]:
mapping.keys()

<KeysViewHDF5 ['cell_identifiers', 'cluster_identifiers', 'correlation', 'metadata', 'votes']>

In [14]:
cluster_identifiers = mapping['cluster_identifiers'][()].astype(str)
cluster_identifiers

array(['CS202210140_1000', 'CS202210140_1001', 'CS202210140_1002', ...,
       'CS202210140_997', 'CS202210140_998', 'CS202210140_999'],
      shape=(3313,), dtype='<U16')

In [15]:
cell_identifiers = mapping['cell_identifiers'][()].astype(str)
cell_identifiers

array(['TTCCTAAGTCTACAAC-436_C04', 'GATCGTACACAGTGTT-155_A01',
       'TCTGGCTGTGCTATTG-166_A01', ..., 'TGCATGAAGACCAACG-155_A01',
       'TGCTTGCCAGGGAGAG-937_B02', 'TGTAACGCACCCAACG-202.1_A01'],
      shape=(5000,), dtype='<U26')

## Convert cluster identifiers to human-readable form

You can use the `TaxonomyTree` class provided by the `cell_type_mapper` to map `cluster_identifiers` to more familiar names at all levels of the taxonomy.

In [16]:
metadata = json.loads(mapping['metadata'][()].decode('utf-8'))

In [17]:
import cell_type_mapper.taxonomy.taxonomy_tree as tree_module

In [18]:
taxonomy_tree = tree_module.TaxonomyTree(data=metadata['taxonomy_tree'])

/Users/scott.daniel/KnowledgeEngineering/cell_type_mapper/src/cell_type_mapper/taxonomy/utils.py:253: UserWarning: This taxonomy has no mapping from leaf_node -> rows in the cell by gene matrix
  warnings.warn("This taxonomy has no mapping from leaf_node -> rows "


In [19]:
rng = np.random.default_rng(88123)
for cl in rng.choice(cluster_identifiers, 5):

    # map leaf node identifier to its human-readable name
    name = taxonomy_tree.label_to_name(level=taxonomy_tree.leaf_level, label=cl)

    # map leaf level to its human-readable name
    level_name = taxonomy_tree.level_to_name(taxonomy_tree.leaf_level)

    result = {
        level_name: name
    }

    # get the parent taxons of this leaf node
    parentage = taxonomy_tree.parents(level=taxonomy_tree.leaf_level, node=cl)

    # mape them to their human-readable names
    result.update({
        taxonomy_tree.level_to_name(level): taxonomy_tree.label_to_name(level=level, label=parentage[level])
        for level in parentage
    })
    print(f'{cl} -- {result}')
    
    

CS202210140_3062 -- {'subcluster': 'ULIT_125_2568', 'cluster': 'ULIT_125', 'supercluster': 'Upper-layer intratelencephalic'}
CS202210140_1676 -- {'subcluster': 'Splat_384_1182', 'cluster': 'Splat_384', 'supercluster': 'Splatter'}
CS202210140_3137 -- {'subcluster': 'DLIT_150_2643', 'cluster': 'DLIT_150', 'supercluster': 'Deep-layer intratelencephalic'}
CS202210140_2230 -- {'subcluster': 'EMSN_426_1736', 'cluster': 'EMSN_426', 'supercluster': 'Eccentric medium spiny neuron'}
CS202210140_733 -- {'subcluster': 'MGE_246_239', 'cluster': 'MGE_246', 'supercluster': 'MGE interneuron'}


In [20]:
votes = mapping['votes'][()]

In [21]:
correlation = mapping['correlation'][()]

## Examine one cell's mapping results

In [22]:
i_row = 0
cell_label = cell_identifiers[i_row]
vv = votes[i_row, :]
corr = correlation[i_row, :]
non_zero = np.where(vv > 0)
print(f'cell: {cell_label}')
for cl, v, c in zip(cluster_identifiers[non_zero], vv[non_zero], corr[non_zero]):
    print(f'subcluster: {cl} -- votes {v} -- correlation {c}')

cell: TTCCTAAGTCTACAAC-436_C04
subcluster: CS202210140_1997 -- votes 1 -- correlation 0.31621815286777505
subcluster: CS202210140_2216 -- votes 1 -- correlation 0.35710822010131493
subcluster: CS202210140_2221 -- votes 1 -- correlation 0.3316740136389966
subcluster: CS202210140_2225 -- votes 1 -- correlation 0.5435488838334297
subcluster: CS202210140_2226 -- votes 6 -- correlation 0.3223849356747381
subcluster: CS202210140_2227 -- votes 6 -- correlation 0.3889955734116062
subcluster: CS202210140_2228 -- votes 1 -- correlation 0.39370598427128195
subcluster: CS202210140_2230 -- votes 1 -- correlation 0.38094404726535624
subcluster: CS202210140_2417 -- votes 1 -- correlation 0.38165855252095726
subcluster: CS202210140_2418 -- votes 1 -- correlation 0.31022976073385367
subcluster: CS202210140_2422 -- votes 1 -- correlation 0.3701507888349749
subcluster: CS202210140_2425 -- votes 1 -- correlation 0.29960363179893157
subcluster: CS202210140_2629 -- votes 63 -- correlation 0.2720083439493894